# Validación Técnica Completa — Sprint 4 Multi-label

**Métricas calculadas sobre el TEST SET (datos no vistos durante entrenamiento):**
- AUC-ROC por clase
- Precisión, Sensibilidad (Recall), Especificidad, F1 por clase
- mAP (mean Average Precision)
- Comparativa con Wang et al. 2017

**Configuración:** Thresholds calibrados por criterio clínico + Temperature scaling T=0.2669


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, json, shutil
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
import numpy as np
import pandas as pd

!pip install -q torchxrayvision h5py scikit-learn
print('Setup OK')

CLASSES_14 = [
    'Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion',
    'Emphysema','Fibrosis','Hernia','Infiltration','Mass','Nodule',
    'Pleural_Thickening','Pneumonia','Pneumothorax',
]

WANG_2017 = {
    'Atelectasis':0.7003,'Cardiomegaly':0.8100,'Consolidation':0.7032,
    'Edema':0.8052,'Effusion':0.7585,'Emphysema':0.8330,'Fibrosis':0.7859,
    'Hernia':0.8717,'Infiltration':0.6614,'Mass':0.6931,'Nodule':0.6693,
    'Pleural_Thickening':0.6835,'Pneumonia':0.6580,'Pneumothorax':0.7993,
}

# Thresholds calibrados
THRESHOLDS = {
    'Atelectasis':0.452,'Cardiomegaly':0.431,'Consolidation':0.247,
    'Edema':0.635,'Effusion':0.577,'Emphysema':0.42,'Fibrosis':0.468,
    'Hernia':0.189,'Infiltration':0.376,'Mass':0.362,'Nodule':0.475,
    'Pleural_Thickening':0.4,'Pneumonia':0.31,'Pneumothorax':0.289,
}

TEMPERATURE = 0.2669

H5_LOCAL     = '/content/nih_images.h5'
PROCESSED_ML = '/content/processed_s4ml'


## 2. Cargar HDF5, CSVs y modelos

In [ ]:
import h5py

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 4_000_000_000):
    print('Copiando HDF5...'); shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)
with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'HDF5: {len(hf["images"]):,} imgs')

os.makedirs(PROCESSED_ML, exist_ok=True)
for csv in ['test.csv']:
    local = f'{PROCESSED_ML}/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s4ml/{csv}', local)
    print(f'{csv}: {len(pd.read_csv(local))} filas')


In [ ]:
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

from src.models.cnn_vit import CNNViT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def load_model(n_layers, ckpt_path):
    m = CNNViT(backbone_weights='densenet121-res224-nih',
               num_classes=14, embedding_dim=512, num_heads=8,
               num_layers=n_layers, mlp_dim=1024).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    return m

model_v1 = load_model(4, f'{PROJECT_ROOT}/experiments_s4ml/checkpoints/sprint4ml_phase2_best.pt')
model_v2 = load_model(6, f'{PROJECT_ROOT}/experiments_s4ml_v2/checkpoints/phase2_best.pt')
print('✓ v1 (4 capas) y v2 (6 capas) cargados')


## 3. Inferencia con ensemble + temperature scaling

In [ ]:
from torch.utils.data import DataLoader
from src.datasets.nih_multilabel import NIHDatasetMultiLabel
from src.datasets.transforms import build_eval_transform

eval_tfm = build_eval_transform(224)
test_ds  = NIHDatasetMultiLabel(f'{PROCESSED_ML}/test.csv', H5_LOCAL, eval_tfm)
loader   = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

print(f'Test set: {len(test_ds)} imágenes')

all_probs, all_labels = [], []
T = TEMPERATURE  # 0.2669

with torch.no_grad():
    for batch in loader:
        imgs = batch['image'].to(device)
        # Temperature scaling aplicado antes del sigmoid
        p1   = torch.sigmoid(model_v1(imgs) / T).cpu().numpy()
        p2   = torch.sigmoid(model_v2(imgs) / T).cpu().numpy()
        # Ensemble ponderado
        probs = 0.3 * p1 + 0.7 * p2
        all_probs.append(probs)
        all_labels.append(batch['label'].numpy())

test_probs  = np.vstack(all_probs)   # (N, 14)
test_labels = np.vstack(all_labels)  # (N, 14)

print(f'Predicciones: {test_probs.shape}')
print(f'Labels:       {test_labels.shape}')
print(f'Temperature T={T} aplicada ✓')


## 4. Métricas completas por clase

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve,
)

results = {}

print('VALIDACIÓN TÉCNICA — TEST SET')
print('='*90)
print(f"{'Clase':<22} {'AUC':>6} {'AP':>6} {'Prec':>6} {'Sens':>6} {'Spec':>6} {'F1':>6} {'TP':>5} {'FP':>5} {'TN':>5} {'FN':>5}")
print('-'*90)

for i, cls in enumerate(CLASSES_14):
    y_true = test_labels[:, i]
    y_prob = test_probs[:, i]
    thr    = THRESHOLDS[cls]
    y_pred = (y_prob >= thr).astype(int)

    n_pos = int(y_true.sum())
    n_neg = int(len(y_true) - n_pos)

    # AUC y AP
    if n_pos > 0 and n_neg > 0:
        auc = roc_auc_score(y_true, y_prob)
        ap  = average_precision_score(y_true, y_prob)
    else:
        auc, ap = float('nan'), float('nan')

    # Confusion matrix
    if y_pred.sum() > 0:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    else:
        tp, fp, tn, fn = 0, 0, n_neg, n_pos

    # Métricas binarias con threshold calibrado
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # recall/sensitivity
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # specificity
    f1   = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0

    results[cls] = {
        'auc': auc, 'ap': ap,
        'precision': prec, 'sensitivity': sens,
        'specificity': spec, 'f1': f1,
        'tp': int(tp), 'fp': int(fp),
        'tn': int(tn), 'fn': int(fn),
        'n_pos': n_pos, 'n_neg': n_neg,
        'threshold': thr,
    }

    auc_str = f'{auc:.4f}' if not np.isnan(auc) else '  N/A'
    ap_str  = f'{ap:.4f}'  if not np.isnan(ap)  else '  N/A'
    marker  = ' ←' if cls in ['Infiltration','Pneumonia'] else ''
    print(f'{cls:<22} {auc_str:>6} {ap_str:>6} {prec:>6.3f} {sens:>6.3f} {spec:>6.3f} {f1:>6.3f} {tp:>5} {fp:>5} {tn:>5} {fn:>5}{marker}')

print('-'*90)

# Macro promedios
valid_auc  = [r['auc']  for r in results.values() if not np.isnan(r['auc'])]
valid_ap   = [r['ap']   for r in results.values() if not np.isnan(r['ap'])]
macro_prec = np.mean([r['precision']   for r in results.values()])
macro_sens = np.mean([r['sensitivity'] for r in results.values()])
macro_spec = np.mean([r['specificity'] for r in results.values()])
macro_f1   = np.mean([r['f1']          for r in results.values()])
macro_auc  = np.mean(valid_auc)
macro_ap   = np.mean(valid_ap)

print(f"{'MACRO AVERAGE':<22} {macro_auc:>6.4f} {macro_ap:>6.4f} {macro_prec:>6.3f} {macro_sens:>6.3f} {macro_spec:>6.3f} {macro_f1:>6.3f}")
print()
print(f'AUC macro:         {macro_auc:.4f}  (Wang 2017: 0.7452  |  Delta: {macro_auc-0.7452:+.4f})')
print(f'mAP:               {macro_ap:.4f}')
print(f'Sensibilidad macro: {macro_sens:.4f}')
print(f'Especificidad macro:{macro_spec:.4f}')
print(f'F1 macro:          {macro_f1:.4f}')
print()
print(f'Clases que superan Wang 2017: {sum(1 for c in CLASSES_14 if not np.isnan(results[c]["auc"]) and results[c]["auc"] > WANG_2017[c])}/14')


## 5. Tabla comparativa con Wang 2017

In [ ]:
print('COMPARATIVA AUC — Ensemble vs Wang et al. 2017:')
print('='*55)
print(f"{'Clase':<22} {'Wang17':>7} {'Ensemble':>9} {'Delta':>8} {'Supera'}")
print('-'*55)

for cls in CLASSES_14:
    wang  = WANG_2017[cls]
    ens   = results[cls]['auc']
    if np.isnan(ens):
        print(f'{cls:<22} {wang:>7.4f} {"N/A":>9} {"N/A":>8}')
        continue
    delta = ens - wang
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
    print(f'{cls:<22} {wang:>7.4f} {ens:>9.4f} {delta:>+8.4f} {arrow}')

wang_macro = np.mean(list(WANG_2017.values()))
print('-'*55)
print(f"{'AUC MACRO':<22} {wang_macro:>7.4f} {macro_auc:>9.4f} {macro_auc-wang_macro:>+8.4f} ↑")


## 6. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import roc_curve

fig_dir = f'{PROJECT_ROOT}/experiments_s4ml_v2/figures'
os.makedirs(fig_dir, exist_ok=True)

# ── Plot 1: AUC por clase (barras comparativas) ───────────────────
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(CLASSES_14))
w = 0.35

wang_vals = [WANG_2017[c] for c in CLASSES_14]
ens_vals  = [results[c]['auc'] if not np.isnan(results[c]['auc']) else 0 for c in CLASSES_14]
colors    = ['#2E7D32' if ens_vals[i] > wang_vals[i] else '#C62828' for i in range(14)]

ax.bar(x - w/2, wang_vals, w, label='Wang et al. 2017', color='#9E9E9E', alpha=0.8)
bars = ax.bar(x + w/2, ens_vals, w, label=f'CNN-ViT Ensemble (macro={macro_auc:.3f})', color=colors, alpha=0.9)

ax.axhline(wang_macro, color='gray', linestyle='--', alpha=0.5,
           label=f'Wang macro={wang_macro:.3f}')
ax.axhline(macro_auc, color='#1565C0', linestyle='--', alpha=0.5,
           label=f'Ensemble macro={macro_auc:.3f}')

ax.set_xticks(x)
ax.set_xticklabels(CLASSES_14, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.5, 1.0)
ax.set_title('AUC-ROC por clase: Wang 2017 vs CNN-ViT Ensemble (test set)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')

green_patch = mpatches.Patch(color='#2E7D32', label='Supera Wang')
red_patch   = mpatches.Patch(color='#C62828', label='Por debajo de Wang')
ax.legend(handles=[
    mpatches.Patch(color='#9E9E9E', label='Wang 2017'),
    mpatches.Patch(color='#1565C0', label=f'CNN-ViT Ensemble (AUC={macro_auc:.3f})'),
    green_patch, red_patch,
], fontsize=9)

plt.tight_layout()
plt.savefig(f'{fig_dir}/auc_comparativa_final.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 2: Sens, Spec, F1 por clase (radar o barras) ────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_to_plot = [
    ('sensitivity', 'Sensibilidad (Recall)', '#1565C0'),
    ('specificity', 'Especificidad',          '#2E7D32'),
    ('f1',          'F1-Score',               '#E65100'),
]

for ax, (key, label, color) in zip(axes, metrics_to_plot):
    vals = [results[c][key] for c in CLASSES_14]
    bars = ax.barh(CLASSES_14, vals, color=color, alpha=0.8)
    ax.axvline(np.mean(vals), color='red', linestyle='--', alpha=0.7,
               label=f'Media={np.mean(vals):.3f}')
    ax.set_xlim(0, 1.05)
    ax.set_xlabel(label)
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis='x')
    # Colorear Infiltration y Pneumonia
    for j, cls in enumerate(CLASSES_14):
        if cls in ['Infiltration', 'Pneumonia']:
            bars[j].set_edgecolor('red')
            bars[j].set_linewidth(2)

plt.suptitle('Métricas por clase — Thresholds calibrados (test set)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{fig_dir}/metrics_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 3: Curvas ROC de todas las clases ────────────────────────
fig, axes = plt.subplots(2, 7, figsize=(20, 8))
axes = axes.flatten()

for i, (cls, ax) in enumerate(zip(CLASSES_14, axes)):
    y_true = test_labels[:, i]
    y_prob = test_probs[:, i]
    if y_true.sum() == 0:
        ax.set_visible(False)
        continue
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = results[cls]['auc']
    ax.plot(fpr, tpr, 'b-', lw=1.5)
    ax.plot([0,1],[0,1],'--',color='gray',alpha=0.4)
    ax.fill_between(fpr, tpr, alpha=0.1)
    ax.set_title(f'{cls}\nAUC={auc:.3f}', fontsize=8)
    ax.set_xlabel('FPR', fontsize=7)
    ax.set_ylabel('TPR', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(alpha=0.3)
    # Marcar threshold calibrado
    thr = THRESHOLDS[cls]
    r = results[cls]
    ax.scatter([1-r['specificity']], [r['sensitivity']],
               s=60, c='red', zorder=5)

plt.suptitle(f'Curvas ROC — CNN-ViT Ensemble (AUC macro={macro_auc:.4f}) — Test Set',
             fontsize=12)
plt.tight_layout()
plt.savefig(f'{fig_dir}/roc_curves_all_classes.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ 3 gráficas guardadas en Drive')


## 7. Guardar métricas completas

In [ ]:
validation_report = {
    "dataset":     "NIH ChestX-ray14 (test set)",
    "n_samples":   len(test_ds),
    "model":       "CNN-ViT Ensemble (v1=4 capas + v2=6 capas)",
    "weights":     {"v1": 0.3, "v2": 0.7},
    "temperature": TEMPERATURE,
    "thresholds":  THRESHOLDS,
    "macro_metrics": {
        "auc_macro":          round(macro_auc,  4),
        "map":                round(macro_ap,   4),
        "precision_macro":    round(macro_prec, 4),
        "sensitivity_macro":  round(macro_sens, 4),
        "specificity_macro":  round(macro_spec, 4),
        "f1_macro":           round(macro_f1,   4),
    },
    "wang_2017_auc_macro":    0.7452,
    "delta_vs_wang":          round(macro_auc - 0.7452, 4),
    "classes_above_wang":     sum(
        1 for c in CLASSES_14
        if not np.isnan(results[c]['auc']) and results[c]['auc'] > WANG_2017[c]
    ),
    "per_class": {
        cls: {k: round(v, 4) if isinstance(v, float) else v
              for k, v in results[cls].items()}
        for cls in CLASSES_14
    }
}

out_path = f'{PROJECT_ROOT}/experiments_s4ml_v2/validation_report.json'
with open(out_path, 'w') as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print(f'✓ Guardado: {out_path}')

print()
print('='*55)
print('RESUMEN FINAL DE VALIDACIÓN TÉCNICA')
print('='*55)
print(f'Dataset:             NIH ChestX-ray14 test set ({len(test_ds)} imágenes)')
print(f'Modelo:              CNN-ViT Ensemble (T={TEMPERATURE})')
print(f'AUC macro:           {macro_auc:.4f}  vs Wang 0.7452  ({macro_auc-0.7452:+.4f})')
print(f'mAP:                 {macro_ap:.4f}')
print(f'Sensibilidad macro:  {macro_sens:.4f}')
print(f'Especificidad macro: {macro_spec:.4f}')
print(f'F1 macro:            {macro_f1:.4f}')
print(f'Clases sobre Wang:   {validation_report["classes_above_wang"]}/14')
print()
print('Clases clave (HNAL):')
for cls in ['Infiltration', 'Pneumonia', 'Cardiomegaly', 'Effusion']:
    r = results[cls]
    print(f'  {cls:<20}: AUC={r["auc"]:.4f} Sens={r["sensitivity"]:.3f} Spec={r["specificity"]:.3f} F1={r["f1"]:.3f}')
